# 09: Left-hand support (C2)

In [2]:
import numpy as np, pandas as pd
from pathlib import Path
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
train = pd.read_csv(ROOT / "data/processed/train.csv")
feat_cols = [f"f{i}" for i in range(63)]
X = train[feat_cols].to_numpy(dtype=np.float64)
y = train["label"].astype(str).to_numpy()
print("shape:", X.shape, "| wrist (f0,f1,f2) max:", np.abs(X[:, :3]).max())

def mirror(X):
    Xm = X.copy()
    Xm[:, 0::3] *= -1          # flip only the x columns
    return Xm

print("mirror twice = original?", np.allclose(mirror(mirror(X)), X))
print("first sample, thumb-tip x (f12): before", X[0, 12].round(3), "after", mirror(X)[0, 12].round(3))

shape: (2165, 63) | wrist (f0,f1,f2) max: 0.0
mirror twice = original? True
first sample, thumb-tip x (f12): before 0.568 after -0.568


In [3]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def make_model():
    return Pipeline([("scaler", StandardScaler()),
                     ("svm", SVC(kernel="rbf", C=100, gamma=0.01))])

rows = []
for fold, (tr, va) in enumerate(skf.split(X, y), start=1):
    Xtr, ytr, Xva, yva = X[tr], y[tr], X[va], y[va]
    Xva_left = mirror(Xva)                      # fake test for the left hand

    m1 = make_model().fit(Xtr, ytr)             # normal training (right hand only)
    m2 = make_model().fit(np.vstack([Xtr, mirror(Xtr)]),
                          np.concatenate([ytr, ytr]))   # (a) mirror-train

    tests = {
        "1 baseline | right hand":       (m1, Xva),
        "2 baseline | left hand (no fix)": (m1, Xva_left),
        "3 (b) flip | left hand":        (m1, mirror(Xva_left)),
        "4 (a) mirror-train | right hand": (m2, Xva),
        "5 (a) mirror-train | left hand":  (m2, Xva_left),
    }
    for name, (m, Xt) in tests.items():
        p = m.predict(Xt)
        rows.append({"fold": fold, "variant": name,
                     "acc": accuracy_score(yva, p),
                     "macro_f1": f1_score(yva, p, average="macro")})

res = pd.DataFrame(rows)
summary = res.groupby("variant")[["acc", "macro_f1"]].mean().round(4)
print(summary)

(ROOT / "results").mkdir(exist_ok=True)
res.to_csv(ROOT / "results/advanced_left_hand_cv.csv", index=False)

                                    acc  macro_f1
variant                                          
1 baseline | right hand          0.9187    0.9142
2 baseline | left hand (no fix)  0.7760    0.7668
3 (b) flip | left hand           0.9187    0.9142
4 (a) mirror-train | right hand  0.9316    0.9279
5 (a) mirror-train | left hand   0.9312    0.9274


In [4]:
piv = res.pivot(index="fold", columns="variant", values="acc")
piv.columns = ["base_R", "base_L", "flip_L", "mir_R", "mir_L"]
piv["mir_R - base_R"] = piv["mir_R"] - piv["base_R"]
print(piv.round(4))
print("mirror-train better in every fold?", (piv["mir_R - base_R"] > 0).all())

      base_R  base_L  flip_L   mir_R   mir_L  mir_R - base_R
fold                                                        
1     0.9145  0.7737  0.9145  0.9353  0.9353          0.0208
2     0.9215  0.8037  0.9215  0.9261  0.9261          0.0046
3     0.8984  0.7436  0.8984  0.9215  0.9192          0.0231
4     0.9238  0.7575  0.9238  0.9353  0.9353          0.0115
5     0.9353  0.8014  0.9353  0.9400  0.9400          0.0046
mirror-train better in every fold? True


In [5]:
import joblib

test = pd.read_csv(ROOT / "data/processed/test.csv")
Xte = test[feat_cols].to_numpy(dtype=np.float64)
yte = test["label"].astype(str).to_numpy()
Xte_left = mirror(Xte)                       # fake test for the left hand

core = joblib.load(ROOT / "models/best_model_core_v1.joblib")   # old model, not touched

final_a = Pipeline([("scaler", StandardScaler()),
                    ("svm", SVC(kernel="rbf", C=100, gamma=0.01,
                                probability=True, random_state=42))])
final_a.fit(np.vstack([X, mirror(X)]), np.concatenate([y, y]))

tests = {
    "baseline core v1 | right":          (core,    Xte),
    "baseline core v1 | left (no fix)":  (core,    Xte_left),
    "(b) flip core v1 | left":           (core,    mirror(Xte_left)),
    "(a) mirror-train v2 | right":       (final_a, Xte),
    "(a) mirror-train v2 | left":        (final_a, Xte_left),
}
out = []
for name, (m, Xt) in tests.items():
    p = m.predict(Xt)
    out.append({"variant": name,
                "test_acc": accuracy_score(yte, p),
                "test_macro_f1": f1_score(yte, p, average="macro")})
out = pd.DataFrame(out).round(4)
print(out.to_string(index=False))

out.to_csv(ROOT / "results/advanced_left_hand_test.csv", index=False)
joblib.dump(final_a, ROOT / "models/best_model_v2_mirror.joblib")

d:\CS 5 Course\ML(Lab)\Project\psl-translator\.venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


                         variant  test_acc  test_macro_f1
        baseline core v1 | right    0.9502         0.9482
baseline core v1 | left (no fix)    0.7972         0.7907
         (b) flip core v1 | left    0.9502         0.9482
     (a) mirror-train v2 | right    0.9377         0.9355
      (a) mirror-train v2 | left    0.9377         0.9355


['d:\\CS 5 Course\\ML(Lab)\\Project\\psl-translator\\models\\best_model_v2_mirror.joblib']